In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

ml = Path("../../data/processed/ml")
X_train = pd.read_parquet(ml / "X_train.parquet")
X_test  = pd.read_parquet(ml / "X_test.parquet")
y_train = pd.read_parquet(ml / "y_train.parquet")["Valeur fonciere"]
y_test  = pd.read_parquet(ml / "y_test.parquet")["Valeur fonciere"]

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
X_train.head()


X_train : (450300, 13)
X_test  : (112576, 13)


,Code departement,Type local,Surface reelle bati,Nombre pieces principales,Surface terrain,annee,mois,code_commune_geo,codeRegion,population_geo,longitude,latitude,revenu_median
481575,27,Appartement,68.0,3.0,0.0,2025,3,27229,28.0,49360.0,1.1406,49.0180,16830.5
35414,30,Maison,135.0,5.0,1194.0,2021,7,30033,76.0,6066.0,4.3222,43.6896,19563.5
149922,13,Appartement,52.0,2.0,0.0,2022,1,13001,93.0,149695.0,5.3879,43.5360,22484.0
45931,34,Maison,54.0,2.0,500.0,2021,2,34300,76.0,5517.0,3.3017,43.4121,18788.6
202762,54,Appartement,72.0,4.0,0.0,2022,6,54304,44.0,14771.0,6.1183,48.6839,20656.4


In [2]:
def ajouter_features_simples(X):
    X = X.copy()

    # Emprise totale (bâti + terrain)
    X["surface_totale"] = X["Surface reelle bati"] + X["Surface terrain"]

    # Taille moyenne d'une pièce (on évite la division par 0)
    nb_pieces = X["Nombre pieces principales"].replace(0, np.nan)
    X["surface_par_piece"] = (X["Surface reelle bati"] / nb_pieces).fillna(0)

    # Temps continu : 2021.0, 2021.083, ... (capte la tendance mensuelle des prix)
    X["date_num"] = X["annee"] + (X["mois"] - 1) / 12

    # Interaction : grande surface DANS une zone à hauts revenus
    X["surface_x_revenu"] = X["Surface reelle bati"] * X["revenu_median"]

    return X

X_train = ajouter_features_simples(X_train)
X_test  = ajouter_features_simples(X_test)

print("Nouvelles colonnes :", X_train.shape[1])
X_train[["surface_totale", "surface_par_piece", "date_num", "surface_x_revenu"]].head()


Nouvelles colonnes : 17


,surface_totale,surface_par_piece,date_num,surface_x_revenu
481575,68.0,22.666667,2025.166667,1144474.0
35414,1329.0,27.000000,2021.500000,2641072.5
149922,52.0,26.000000,2022.000000,1169168.0
45931,554.0,27.000000,2021.083333,1014584.4
202762,72.0,18.000000,2022.416667,1487260.8


In [3]:
def distance_haversine(lat1, lon1, lat2, lon2):
    """Distance en km entre deux points GPS (tient compte de la courbure de la Terre)."""
    R = 6371  # rayon de la Terre en km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

# Coordonnées du centre de Paris (Notre-Dame)
PARIS_LAT, PARIS_LON = 48.8566, 2.3522


In [4]:
def ajouter_features_geo(X):
    X = X.copy()

    # Distance au centre de Paris (proxy d'attractivité nationale)
    X["distance_paris"] = distance_haversine(
        X["latitude"], X["longitude"], PARIS_LAT, PARIS_LON
    )

    # Zones premium (on compare en texte pour gérer "75" et 75)
    dept = X["Code departement"].astype(str)
    X["is_paris"] = (dept == "75").astype(int)
    X["is_petite_couronne"] = dept.isin(["75", "92", "93", "94"]).astype(int)

    return X

X_train = ajouter_features_geo(X_train)
X_test  = ajouter_features_geo(X_test)

print("Colonnes :", X_train.shape[1])
X_train[["distance_paris", "is_paris", "is_petite_couronne"]].describe()


Colonnes : 20


,distance_paris,is_paris,is_petite_couronne
count,450300.000000,450300.000000,450300.000000
mean,411.142045,0.029458,0.076180
std,793.925530,0.169087,0.265287
min,0.458399,0.000000,0.000000
25%,172.714672,0.000000,0.000000
50%,346.888000,0.000000,0.000000
75%,512.060660,0.000000,0.000000
max,9421.313960,1.000000,1.000000


In [5]:
# 1) On calcule la table dept -> prix médian sur le TRAIN UNIQUEMENT
dept_train = X_train["Code departement"].astype(str)
prix_par_dept = y_train.groupby(dept_train).median()

# 2) Valeur de repli si un département n'existe pas dans le train
global_median = y_train.median()

# 3) On applique cette table (celle du train) aux DEUX jeux
def encoder_prix_dept(X):
    X = X.copy()
    dept = X["Code departement"].astype(str)
    X["prix_median_dept"] = dept.map(prix_par_dept).fillna(global_median)
    return X

X_train = encoder_prix_dept(X_train)
X_test  = encoder_prix_dept(X_test)

print("Colonnes :", X_train.shape[1])
X_train[["Code departement", "prix_median_dept"]].head()


Colonnes : 21


,Code departement,prix_median_dept
481575,27,183150.0
35414,30,200000.0
149922,13,241538.5
45931,34,188255.0
202762,54,154000.0


Le raisonnement (très important) :

On regroupe par département et on prend le prix médian — mais seulement sur y_train.
La table prix_par_dept (apprise sur le train) est ensuite appliquée au test via .map(). Le test n'a donc jamais servi à calculer ces médianes → pas de fuite.
fillna(global_median) : si un département apparaissait dans le test sans être dans le train, on met la médiane globale (filet de sécurité).
🚨 C'est exactement le même principe que la médiane du terrain (notebook 02) : toute statistique qui touche à la cible ou sert de référence se calcule sur le train, s'applique au test. Tu commences à avoir le réflexe. 💪

🎓 Pour aller plus loin (à mentionner à l'oral) : la version « pro » de cette technique utilise un encodage par K-folds (out-of-fold) pour éviter qu'une ligne du train ne s'auto-influence. Ici, comme chaque département a des milliers de biens, l'influence d'une seule ligne est négligeable → notre version simple est acceptable. Mais savoir que le K-fold existe montre que tu maîtrises le sujet.

In [6]:
from sklearn.model_selection import KFold

def target_encode_kfold(cle_train, cible, cle_test, n_splits=5, m=20, seed=42):
    """
    Encodage par la cible OUT-OF-FOLD (anti-surapprentissage).
    - Train : chaque ligne encodée sans utiliser son propre fold.
    - Test  : encodé avec tout le train.
    """
    moyenne_globale = cible.mean()

    # --- TRAIN : out-of-fold ---
    oof = pd.Series(index=cle_train.index, dtype=float)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for idx_calc, idx_encode in kf.split(cle_train):
        # on calcule les moyennes sur idx_calc, on encode idx_encode
        k = cle_train.iloc[idx_calc]
        t = cible.iloc[idx_calc]
        agg = t.groupby(k).agg(["mean", "count"])
        lisse = (agg["count"] * agg["mean"] + m * moyenne_globale) / (agg["count"] + m)
        oof.iloc[idx_encode] = cle_train.iloc[idx_encode].map(lisse).fillna(moyenne_globale).values

    # --- TEST : calculé sur TOUT le train ---
    agg_full = cible.groupby(cle_train).agg(["mean", "count"])
    lisse_full = (agg_full["count"] * agg_full["mean"] + m * moyenne_globale) / (agg_full["count"] + m)
    enc_test = cle_test.map(lisse_full).fillna(moyenne_globale)

    return oof, enc_test

X_train["prix_median_commune"], X_test["prix_median_commune"] = target_encode_kfold(
    X_train["code_commune_geo"],
    np.log1p(y_train),          # toujours sur le log (robuste)
    X_test["code_commune_geo"],
    n_splits=5, m=20
)

print("Colonnes :", X_train.shape[1])
print("Corrélation avec log(prix) :",
      np.corrcoef(X_train["prix_median_commune"], np.log1p(y_train))[0, 1].round(3))


Colonnes : 22
Corrélation avec log(prix) : 0.482


In [7]:
from pathlib import Path

ml = Path("../../data/processed/ml")
X_train.to_parquet(ml / "X_train_fe.parquet")
X_test.to_parquet(ml / "X_test_fe.parquet")

print("Features sauvegardées ✅")
print("X_train :", X_train.shape, "| X_test :", X_test.shape)


Features sauvegardées ✅
X_train : (450300, 22) | X_test : (112576, 22)


In [8]:
print(f"{X_train.shape[1]} colonnes :\n")
for c in X_train.columns:
    print(" -", c)


22 colonnes :

 - Code departement
 - Type local
 - Surface reelle bati
 - Nombre pieces principales
 - Surface terrain
 - annee
 - mois
 - code_commune_geo
 - codeRegion
 - population_geo
 - longitude
 - latitude
 - revenu_median
 - surface_totale
 - surface_par_piece
 - date_num
 - surface_x_revenu
 - distance_paris
 - is_paris
 - is_petite_couronne
 - prix_median_dept
 - prix_median_commune


## 🧾 Conclusions du Feature Engineering

12 → 20 features. Nouvelles variables créées :
- **Surfaces** : surface_totale, surface_par_piece
- **Temporel** : date_num (temps continu)
- **Interaction** : surface_x_revenu
- **Géo** : distance_paris (Haversine), is_paris, is_petite_couronne
- **Encodage cible** : prix_median_dept (médiane du train, sans fuite)

Toutes les features ligne-par-ligne sont calculées identiquement train/test.
prix_median_dept est appris sur le train uniquement (règle anti-fuite).
